# 개별종목 조합G — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합G 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합G의 피처 값만 지정합니다.
import json

COMBINATION = 'G'
FEATURE_COLUMNS = (
    'dist_high_60',
    'sma_gap_20_60',
    'relative_ret_5_market',
    'rsi_14',
    'hv_20',
    'turnover_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 24개 노트북이 각각 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20100331 ~ 20240822
학습 행·종목: 171557 162
조합G 피처: ('dist_high_60', 'sma_gap_20_60', 'relative_ret_5_market', 'rsi_14', 'hv_20', 'turnover_20')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130410,20130705,0.3573,0.3701,-0.0128,0.3497,0.2615,0.3164
1,2,balanced,999,20140414,20140711,0.4074,0.4741,-0.0667,0.3568,0.2403,0.3185
2,3,balanced,1248,20150421,20150716,0.3611,0.3330,0.0282,0.3606,0.3299,0.3499
3,4,balanced,1496,20160422,20160719,0.3695,0.4128,-0.0433,0.3585,0.3228,0.3491
4,5,balanced,1745,20170424,20170721,0.3614,0.4182,-0.0568,0.3405,0.3379,0.3463
5,6,balanced,1994,20180503,20180731,0.3789,0.3912,-0.0124,0.3769,0.3374,0.3633
6,7,balanced,2243,20190514,20190806,0.3826,0.4615,-0.0790,0.3506,0.2341,0.3081
7,8,NaN,2492,20200518,20200807,0.3585,0.3144,0.0441,0.3575,0.4368,0.3809
8,9,NaN,2741,20210518,20210810,0.3739,0.4423,-0.0684,0.3558,0.3263,0.3509
9,10,balanced,2989,20220519,20220812,0.3677,0.3343,0.0333,0.3676,0.3451,0.3598


,OOS 폴드 평균
accuracy,0.3727
training_majority_baseline_accuracy,0.3851
accuracy_minus_training_majority_baseline,-0.0124
macro_f1,0.3602
down_recall,0.3301
core_harmonic_mean,0.3505


재실행 명령: python scripts/run_stock_model_experiment.py
